## 0. What this notebook is

Viva walkthrough of **our** Phase-1 system, laid out like the official starter.

The starter used `GPT2Config` + the GPT-2 tokenizer. We did **not**. We trained a custom byte-level BPE (vocab 8192) and four from-scratch models in `phase1_models.py`. None of them uses `nn.Transformer` or a public checkpoint.

Set `RUN_TRAINING = False` during the viva. The architecture cells still run.


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path.cwd().resolve()
SEARCH = [HERE, HERE.parent, Path("/kaggle/working")]
for base in (HERE, HERE.parent, HERE.parent.parent, HERE.parent.parent.parent):
    SEARCH.append(base)
    for phase in range(1, 8):
        SEARCH.append(base / f"phase{phase}")
        SEARCH.append(base / f"phase{phase}" / "code")
        SEARCH.append(base / "code" / f"phase{phase}")
for folder in SEARCH:
    if folder.is_dir() and str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
print("python path heads:")
for folder in sys.path[:8]:
    print(" ", folder)


## 1. Hugging Face login

Needed only if you re-download the CS-781 dataset. Skip this cell on a laptop that already has the JSON.


In [ ]:
import os

try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if HF_TOKEN:
        from huggingface_hub import login
        login(token=HF_TOKEN)
        print("Hugging Face login from environment.")
    else:
        print("No HF token in this session (fine for a viva walkthrough).")
        print(type(exc).__name__, str(exc)[:120])


## 2. Check GPU


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))


## 3. Configuration

These are the numbers of the **selected** zip, not the starter defaults (starter had GPT-2 vocab, block 2048, width 560).


In [ ]:
import os
import random
import json
import math
import zipfile
from pathlib import Path

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = {
    "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
    "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
    "tribunals": "Phase_1_Pretraining/Train/train_tribunals.json",
    "acts": "Phase_1_Pretraining/Train/train_acts.json",
}
# Wise mix: test is SC-style, so SC is the majority; other courts still enter.
TOKEN_MIX = {"supreme_court": 0.50, "high_courts": 0.25, "tribunals": 0.15, "acts": 0.10}
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Tokenizer (ours)
BPE_VOCAB_SIZE = 8192
BPE_MIN_PAIR_COUNT = 2
BPE_SCORE = "dice"

# Selected backbone: efficient
ARCH = "efficient"
BLOCK = 1024
N_EMBD = 1024
N_LAYER = 10
N_HEAD = 16
N_KV_HEAD = 4          # grouped-query attention

# Training
TRAIN_TOKENS = 100_000_000
VAL_TOKENS = 400_000
BATCH = 1
GRAD_ACCUM = 8
EPOCHS = 1
LR = 3e-4

# Generation: greedy, stop at EOS, then clip to 128 words
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 400
TEMPERATURE = 0.0      # greedy. Starter used 0.1 sampling.

RUN_TRAINING = False   # True only on Kaggle T4
RUN_DATA = False

OUTPUT_DIR = Path(os.environ.get("P1_OUT", "viva_output_phase1"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Architecture:", ARCH)
print("Output:", OUTPUT_DIR.resolve())


## 4. Load tokenizer — custom Legal BPE, not GPT-2

Assignment-1 pair-count BPE fills the vocab with pieces of `the` / `of`. We merge only pairs seen at least twice, ranked by Dice:

`score(x,y) = 2 C(xy)^2 / (C(x)+C(y))`

Bytes 0–255 are always in the vocab. A word-end marker keeps whole legal tokens together.


In [ ]:
from legal_bpe import LegalBPE, iter_texts, END, SPECIALS

print("specials", SPECIALS)
print("word-end marker id (internal)", END)
print("Dice training lives on LegalBPE.train / merge scoring in legal_bpe.py")

bpe_candidates = [
    Path(r"C:\Users\Anshuman Raj\CS781\kaggle_efficient2\cs781_phase1_efficient\legal_bpe.json"),
    OUTPUT_DIR / "legal_bpe.json",
    Path("/kaggle/working/cs781_phase1_efficient/legal_bpe.json"),
]
tokenizer = None
for path in bpe_candidates:
    if path.is_file():
        tokenizer = LegalBPE.load(str(path))
        print("loaded", path, "vocab", len(tokenizer))
        break
if tokenizer is None:
    print("No saved BPE on disk. Training one needs RUN_DATA=True and the HF JSON.")


## 5. Encode a legal string

Spaces and case are kept, so Phase-2 character offsets can be inverted.


In [ ]:
samples = [
    "The Supreme Court held that the petitioner",
    "AIR 1995 SC 123",
    "Section 302 of the Indian Penal Code",
]
if tokenizer is None:
    print("skip encode: tokenizer not loaded")
else:
    for text in samples:
        ids = tokenizer.encode(text, add_special_tokens=False)
        print(len(ids), "ids |", text)
        print(" ", ids[:24])
        print(" ", tokenizer.decode(ids))


## 6. Four architectures

All share tied input/output embeddings and block 1024. `build_model` refuses a count outside 50M–300M.

| name | attention / positions | MLP / norm | size |
|---|---|---|---|
| decoder | full causal MHA, learned pos | GELU, LayerNorm | 110.2M |
| hybrid | 256-token window on 3/4 layers | same as decoder | 110.2M |
| efficient | RoPE, GQA 16Q/4KV | SwiGLU, RMSNorm | 118.5M |
| encdec | 6 bi-encoder + 6 causal+cross | GELU, LayerNorm | 106.3M |

Selected zip: **efficient**, greedy + EOS.


In [ ]:
import inspect
from phase1_models import (
    DecoderLM, EfficientLM, EncoderDecoderLM,
    GroupedQueryAttention, apply_rope, build_model, count_parameters,
)

print("=== EfficientLM ===")
print(inspect.getsource(EfficientLM))
print("=== GQA (RoPE on Q and K, 16 query heads share 4 KV heads) ===")
print(inspect.getsource(GroupedQueryAttention)[:1800])


## 7. Hybrid window vs full decoder

Hybrid is `DecoderLM(..., windowed=True)`: layers 0,1,2,4,... see 256 earlier tokens; every 4th layer is full causal so a citation can travel.


In [ ]:
print(inspect.getsource(DecoderLM))


## 8. Encoder–decoder split

Training splits a 1024-token block in half: first half encoded, second half decoded with cross-attention. Phase 6 does **not** reuse that split: it encodes the judgment and decodes the summary.


In [ ]:
print(inspect.getsource(EncoderDecoderLM.forward))


## 9. Instantiate the selected model

Skip building all four at once on a laptop (≈2 GB). One efficient body is enough to show the 50M–300M check.


In [ ]:
BUILD = False  # True only if you want to allocate ~118M parameters

if BUILD:
    model = build_model(ARCH)
    print(ARCH, f"{count_parameters(model)/1e6:.2f}M")
else:
    print("BUILD=False (viva). Sizes from the spec:")
    for name, size in [("decoder", 110.18), ("hybrid", 110.18), ("efficient", 118.49), ("encdec", 106.26)]:
        print(f"  {name:10s} {size:.2f}M")


## 10. Training (Kaggle T4)

Next-token cross-entropy, AdamW, block 1024, Supreme Court JSON. This is the same loop as `phase1_run.py`. Do not run it in the viva.


In [ ]:
print("RUN_TRAINING", RUN_TRAINING)
print("Sources", list(TRAIN_FILES))
print("Token mix", TOKEN_MIX)
print("Real training: phase1_run.py then phase1_mix.py (round-robin the four dumps).")
print("Selected zip: about 116M tokens, 50/25/15/10 SC / HC / tribunals / acts.")
if RUN_TRAINING:
    raise SystemExit("Wire this cell to phase1_run.main('efficient') on Kaggle, not on the viva laptop.")


## 11. Generation: 128 words, greedy + EOS

Starter sampled at T=0.1 and cut 128 words. We decode greedily and stop at `</s>`, then clip to 128 words. Forcing a 128-word pad was worse on BERTScore.


In [ ]:
print("temperature", TEMPERATURE, "(0 = argmax)")
print("stop at EOS, then keep at most", TARGET_GENERATION_WORDS, "words")
print("implementation: phase1_fastgen.py (KV cache) wrapping phase1_models.generate")


## 12. Codabench zip


In [ ]:
print("prediction.json format: {id: {text: continuation}}")
print("zip root must contain only prediction.json")
print("Selected local copy: capstone/phase1/local_runs/best_08328/submission.zip")
